In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

## task description

there are 350000 photos, but only 35000 are labeled with MGS. 

filter out those labeled photos.

what is the sample distribution of those photos?

the MouseGrimaceFaces_main.csv has columns: 
index, id, subset
examples are : 000001.jpg, 6, AW

how many id there is? - you can know by filter out unique id
how many subset there is? - 5 subset, they are AW, JW, KH, LW and MR
under each subset, how many id there are? and which photos belong to those id?
what is the relationship between datasets?

how to use Yolo?

how to use DeepLabCut?

can you do a full image classfification first?

## data observation

In [4]:
main = pd.read_csv("mouse_dataset/MouseGrimaceFaces_main.csv")
mgs = pd.read_csv("mouse_dataset/MouseGrimaceFaces_mgs.csv")

print(main.shape)
print(mgs.shape)

print(main.head())
print(mgs.head())

(34935, 3)
(3406, 57)
        index   id subset
0  000001.jpg    6     AW
1  000002.jpg    6     KH
2  000003.jpg   78     KH
3  000004.jpg  102     KH
4  000005.jpg   30     AW
        index  ot1  nb1  cb1  ep1  wc1 ot2 nb2 cb2 ep2  ... nb11 cb11 ep11  \
0  000008.jpg    -    -    -    -    -   -   -   -   -  ...    1    0    0   
1  000009.jpg    1    1    0    1    2   0   0   0   0  ...    -    -    -   
2  000015.jpg  NaN  NaN  NaN  NaN  NaN   -   -   -   -  ...  NaN  NaN  NaN   
3  000027.jpg  NaN  NaN  NaN  NaN  NaN   1   1   1   1  ...  NaN  NaN  NaN   
4  000031.jpg    -    -    -    -    -   -   -   -   -  ...    1    0    0   

  wc11 ot12 nb12 cb12 ep12 wc12 subset  
0    0    0    1    0    1    1     KH  
1    -    -    -    -    -    -     KH  
2  NaN  NaN  NaN  NaN  NaN  NaN     JW  
3  NaN  NaN  NaN  NaN  NaN  NaN     JW  
4    0    0    1    1    1    1     KH  

[5 rows x 57 columns]


In [5]:
# Check subset imbalance
print(main["subset"].value_counts())
print(mgs["subset"].value_counts())

subset
KH    28112
JW     2778
AW     2771
MR      677
LW      597
Name: count, dtype: int64
subset
KH    1224
MR     677
LW     597
JW     546
AW     362
Name: count, dtype: int64


In [6]:
# Check mouse identity distribution
print(main.groupby("subset")["id"].nunique())

images_per_mouse = main.groupby(["subset", "id"]).size()
print(images_per_mouse.describe())

subset
AW     20
JW     40
KH    126
LW    130
MR     12
Name: id, dtype: int64
count    328.000000
mean     106.509146
std      164.289313
min        3.000000
25%        6.000000
50%       56.000000
75%      101.750000
max      928.000000
dtype: float64


In [7]:
# Check label coverage
labeled_indices = set(mgs["index"])
main["has_mgs_label"] = main["index"].isin(labeled_indices)

print(main["has_mgs_label"].value_counts())
print(main.groupby("subset")["has_mgs_label"].mean())

has_mgs_label
False    31529
True      3406
Name: count, dtype: int64
subset
AW    0.130639
JW    0.196544
KH    0.043540
LW    1.000000
MR    1.000000
Name: has_mgs_label, dtype: float64


In [8]:
# Inspect label values
label_cols = [c for c in mgs.columns if c not in ["index", "subset"]]

for col in label_cols[:10]:
    print(col)
    print(mgs[col].value_counts(dropna=False))

ot1
ot1
0      952
NaN    908
-      772
1      458
2      295
9       21
Name: count, dtype: int64
nb1
nb1
NaN    908
0      847
-      772
1      627
9      170
2       82
Name: count, dtype: int64
cb1
cb1
0      926
NaN    908
-      772
1      549
9      170
2       81
Name: count, dtype: int64
ep1
ep1
1      927
NaN    908
-      772
0      486
2      302
9       10
3        1
Name: count, dtype: int64
wc1
wc1
NaN    908
-      772
0      769
1      696
9      142
2      119
Name: count, dtype: int64
ot2
ot2
-    1730
0    1109
1     368
2     121
9      78
Name: count, dtype: int64
nb2
nb2
-    1730
0     824
9     466
1     356
2      30
Name: count, dtype: int64
cb2
cb2
-    1730
0     915
9     360
1     352
2      49
Name: count, dtype: int64
ep2
ep2
-    1730
0     913
1     567
2     144
9      52
Name: count, dtype: int64
wc2
wc2
-    1730
0     801
9     421
1     365
2      89
Name: count, dtype: int64


## Full-image Klassification

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm


In [ ]:
# folder where current script exists
PROJECT_DIR = Path.cwd()

BASE_DIR = PROJECT_DIR / "mouse_dataset"

IMG_DIR = BASE_DIR / "images"

MGS_CSV = BASE_DIR / "MouseGrimaceFaces_mgs.csv"
MAIN_CSV = BASE_DIR / "MouseGrimaceFaces_main.csv"

print("Current directory:", PROJECT_DIR)
print("Dataset path:", BASE_DIR)
print("MGS exists:", MGS_CSV.exists())
print("Image folder exists:", IMG_DIR.exists())

Current directory: /Users/baturu/Documents/Project_CV
Dataset path: /Users/baturu/Documents/Project_CV/mouse_dataset
MGS exists: True
Image folder exists: True


In [ ]:

def build_labels():
    df = pd.read_csv(MGS_CSV)
    main_df = pd.read_csv(MAIN_CSV)

    # merge id information
    df = df.merge(
        main_df[["index","id"]],
        on="index",
        how="left"
    )

    meta_cols = ["index", "subset", "id"]
    score_cols = [c for c in df.columns if c not in meta_cols]

    scores = df[score_cols].replace("-", np.nan)
    scores = scores.apply(pd.to_numeric, errors="coerce")

    # 9 means "cannot be judged"
    scores = scores.replace(9, np.nan)

    # Average all available MGS scores
    df["mgs_mean"] = scores.mean(axis=1)

    # Remove images without valid scores
    df = df.dropna(subset=["mgs_mean"])

    # Binary label:
    # 0 = well-being / not impaired
    # 1 = impaired / pain-like expression
    df["label"] = (df["mgs_mean"] >= 1.0).astype(int)

    # Build image path
    df["path"] = df["index"].apply(lambda x: IMG_DIR / str(x))

    # Keep only existing images
    df = df[df["path"].apply(lambda p: p.exists())]

    return df[["index", "subset", "id", "path", "mgs_mean", "label"]]


class MouseDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row["label"], dtype=torch.long)

        return img, label


def make_group_split(df):
    groups = df["subset"].astype(str) + "_" + df["id"].astype(str)

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, val_idx = next(
        splitter.split(df, y=df["label"], groups=groups)
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    return train_df, val_df


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc="Training"):
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validation"):
            imgs = imgs.to(device)

            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return all_labels, all_preds


def main():
    df = build_labels()

    print("Total labeled usable images:", len(df))
    print("\nLabel distribution:")
    print(df["label"].value_counts())

    print("\nSubset distribution:")
    print(df["subset"].value_counts())

    train_df, val_df = make_group_split(df)

    print("\nTrain size:", len(train_df))
    print("Validation size:", len(val_df))

    print("\nTrain label distribution:")
    print(train_df["label"].value_counts())

    print("\nValidation label distribution:")
    print(val_df["label"].value_counts())

    train_tfms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_tfms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_ds = MouseDataset(train_df, transform=train_tfms)
    val_ds = MouseDataset(val_df, transform=val_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=32,
        shuffle=False,
        num_workers=0
    )

    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print("\nUsing device:", device)

    model = models.convnext_tiny(
        weights=models.ConvNeXt_Tiny_Weights.DEFAULT
    )

    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features,
        2
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    epochs = 10

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        y_true, y_pred = evaluate(model, val_loader, device)

        print("Train loss:", train_loss)
        print("\nConfusion matrix:")
        print(confusion_matrix(y_true, y_pred))

        print("\nClassification report:")
        print(
            classification_report(
                y_true,
                y_pred,
                target_names=["well-being", "impaired"]
            )
        )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label_mapping": {
                0: "well-being",
                1: "impaired"
            }
        },
        "mouse_wellbeing_convnext_tiny.pt"
    )

    print("\nSaved model to mouse_wellbeing_convnext_tiny.pt")


if __name__ == "__main__":
    main()

Total labeled usable images: 3406

Label distribution:
label
0    2682
1     724
Name: count, dtype: int64

Subset distribution:
subset
KH    1224
MR     677
LW     597
JW     546
AW     362
Name: count, dtype: int64

Train size: 2580
Validation size: 826

Train label distribution:
label
0    2058
1     522
Name: count, dtype: int64

Validation label distribution:
label
0    624
1    202
Name: count, dtype: int64

Using device: mps
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /Users/baturu/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:08<00:00, 13.0MB/s] 



Epoch 1/10


Validation: 100%|██████████| 26/26 [00:44<00:00,  1.71s/it]


Train loss: 0.37197800753293214

Confusion matrix:
[[576  48]
 [ 64 138]]

Classification report:
              precision    recall  f1-score   support

  well-being       0.90      0.92      0.91       624
    impaired       0.74      0.68      0.71       202

    accuracy                           0.86       826
   macro avg       0.82      0.80      0.81       826
weighted avg       0.86      0.86      0.86       826


Epoch 2/10


...

Train loss: 0.07230643120904763

Confusion matrix:
[[600  24]
 [ 79 123]]

Classification report:
              precision    recall  f1-score   support

  well-being       0.88      0.96      0.92       624
    impaired       0.84      0.61      0.70       202

    accuracy                           0.88       826
   macro avg       0.86      0.79      0.81       826
weighted avg       0.87      0.88      0.87       826


Epoch 9/10


Validation: 100%|██████████| 26/26 [00:21<00:00,  1.22it/s]


Train loss: 0.057970684915666044

Confusion matrix:
[[545  79]
 [ 28 174]]

Classification report:
              precision    recall  f1-score   support

  well-being       0.95      0.87      0.91       624
    impaired       0.69      0.86      0.76       202

    accuracy                           0.87       826
   macro avg       0.82      0.87      0.84       826
weighted avg       0.89      0.87      0.87       826


Epoch 10/10


Validation: 100%|██████████| 26/26 [00:22<00:00,  1.17it/s]


Train loss: 0.04368690469296488

Confusion matrix:
[[564  60]
 [ 42 160]]

Classification report:
              precision    recall  f1-score   support

  well-being       0.93      0.90      0.92       624
    impaired       0.73      0.79      0.76       202

    accuracy                           0.88       826
   macro avg       0.83      0.85      0.84       826
weighted avg       0.88      0.88      0.88       826


Saved model to mouse_wellbeing_convnext_tiny.pt


## with DCL Help